In [1]:
import torch

# Step 1: Load the Data

In [2]:
words = open('names.txt', 'r').read().splitlines()
# ['emma', 'olivia', 'ava', ...] — 32,033 names

# Step 2: Count Bigrams (Character Pairs)

We treat each name as a sequence of characters with special start/end tokens:

"emma"  →  . e m m a .

**We count every adjacent pair:**

In [3]:
b = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']  # <S>=start, <E>=end
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

In [4]:
b

{('<S>', 'e'): 1531,
 ('e', 'm'): 769,
 ('m', 'm'): 168,
 ('m', 'a'): 2590,
 ('a', '<E>'): 6640,
 ('<S>', 'o'): 394,
 ('o', 'l'): 619,
 ('l', 'i'): 2480,
 ('i', 'v'): 269,
 ('v', 'i'): 911,
 ('i', 'a'): 2445,
 ('<S>', 'a'): 4410,
 ('a', 'v'): 834,
 ('v', 'a'): 642,
 ('<S>', 'i'): 591,
 ('i', 's'): 1316,
 ('s', 'a'): 1201,
 ('a', 'b'): 541,
 ('b', 'e'): 655,
 ('e', 'l'): 3248,
 ('l', 'l'): 1345,
 ('l', 'a'): 2623,
 ('<S>', 's'): 2055,
 ('s', 'o'): 531,
 ('o', 'p'): 95,
 ('p', 'h'): 204,
 ('h', 'i'): 729,
 ('<S>', 'c'): 1542,
 ('c', 'h'): 664,
 ('h', 'a'): 2244,
 ('a', 'r'): 3264,
 ('r', 'l'): 413,
 ('l', 'o'): 692,
 ('o', 't'): 118,
 ('t', 't'): 374,
 ('t', 'e'): 716,
 ('e', '<E>'): 3983,
 ('<S>', 'm'): 2538,
 ('m', 'i'): 1256,
 ('a', 'm'): 1634,
 ('m', 'e'): 818,
 ('<S>', 'h'): 874,
 ('r', 'p'): 14,
 ('p', 'e'): 197,
 ('e', 'r'): 1958,
 ('r', '<E>'): 1377,
 ('e', 'v'): 463,
 ('v', 'e'): 568,
 ('l', 'y'): 1588,
 ('y', 'n'): 1826,
 ('n', '<E>'): 6763,
 ('b', 'i'): 217,
 ('i', 'g'): 428,


# Step 3: Build a Tensor of Counts (N)


**Instead of a dict, we use a 27×27 matrix (26 letters + 1 for the . boundary token).**

N is a 27×27 table of counts. Each cell N[i, j] answers:

"How many times did character i appear immediately followed by character j in all the names?"


In [5]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}  # string → int
stoi['.'] = 0                                # '.' is index 0
itos = {i:s for s,i in stoi.items()}         # int → string

N = torch.zeros((27, 27), dtype=torch.int32)
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

# Step 4: Convert Counts to Probabilities


In [6]:
P = (N + 1).float()          # +1 smoothing (avoids log(0))
P = P / P.sum(1, keepdim=True)  # normalize each row to sum to 1

# Step 5: Neural Network Pipeline

In [7]:
# 5.a Build the training set

xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)  # input characters (indices 0-26)
ys = torch.tensor(ys)  # target next characters
num = xs.nelement()    # 228,146 total bigrams

In [9]:
# 5.b The model — one-hot → logits → softmax
import torch.nn.functional as F
W = torch.randn((27, 27), requires_grad=True)

for k in range(100):
    # Forward pass
    xenc = F.one_hot(xs, num_classes=27).float()  # (num, 27)
    logits = xenc @ W                              # (num, 27)
    counts = logits.exp()                          # softmax numerator
    probs = counts / counts.sum(1, keepdims=True)  # softmax denominator

    # Loss: negative log-likelihood of the correct next character
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
    print(loss.item())
    
    # Backward pass
    W.grad = None
    loss.backward()

    # Update
    W.data += -50 * W.grad

3.7099251747131348
3.352163791656494
3.152334213256836
3.023606300354004
2.9310240745544434
2.8617889881134033
2.8090879917144775
2.7683944702148438
2.736363172531128
2.7105021476745605
2.6890721321105957
2.670926332473755
2.6553070545196533
2.6416983604431152
2.629730701446533
2.6191301345825195
2.6096858978271484
2.601231336593628
2.593632221221924
2.5867793560028076
2.5805797576904297
2.574955701828003
2.569838762283325
2.565171241760254
2.5609002113342285
2.556980848312378
2.5533740520477295
2.5500433444976807
2.54695987701416
2.5440969467163086
2.541431188583374
2.538942575454712
2.536614179611206
2.534430742263794
2.532378911972046
2.5304479598999023
2.5286264419555664
2.5269064903259277
2.5252790451049805
2.5237386226654053
2.522277355194092
2.5208895206451416
2.51957106590271
2.5183165073394775
2.5171213150024414
2.515982151031494
2.51489520072937
2.51385760307312
2.5128653049468994
2.511916399002075
2.51100754737854
2.5101370811462402
2.509303092956543
2.50850248336792
2.50773

In [10]:
for i in range(5):     # generate 5 names
    out = []
    ix = 0             # start at '.'
    while True:
        # BEFORE (counting version, commented out):
        # p = P[ix]

        # NOW (neural version):
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdims=True)

        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))

NameError: name 'g' is not defined